# 🛡️ RISKTWIN · Piloto de la arquitectura de integración de datos de riesgo industrial

**Notebook piloto (prueba de concepto) en Google Colab**

**Tema de tesis:** *Arquitectura de software para la integración de datos fragmentados de riesgo industrial en un gemelo digital multimodal*
**Autores:** Delgado Quiñonez Elian Adonis · Vargas Ordoñez Stalyn Gabriel
**Institución:** Universidad de Guayaquil · Facultad de Ingeniería Industrial · Sistemas de la Información

---

## ¿Qué demuestra este notebook?

En RISKTWIN, los componentes **SI-01** (indicadores de riesgo) y **SI-02** (dataset de incidentes) se desarrollan por separado y **no comparten un modelo de datos común**. Este notebook construye, en pequeño y de punta a punta, la **capa de integración** que propone la tesis:

```
SI-01 ──┐
        ├─► ETL (normaliza y valida) ─► Base de datos centralizada ─► API REST (OpenAPI) ─┬─► IND-SIS-02 (dashboard)
SI-02 ──┘                                                                                 └─► SIS-API-01 (gemelo digital)
```

### Relación con los objetivos específicos del anteproyecto

| Objetivo específico | Dónde se aborda en este notebook |
|---|---|
| **OE1.** Diseñar un modelo de datos que integre SI-01 y SI-02 | Pasos 3 y 4 (análisis de fuentes y modelo entidad-relación) |
| **OE2.** Implementar una base de datos centralizada | Paso 5 (ETL y carga en SQL) |
| **OE3.** Documentar la arquitectura (componentes, ER, OpenAPI/Swagger) | Pasos 1, 4 y 7 |
| **OE4.** Validar con pruebas de integración | Paso 8 (tiempo de respuesta y tasa de éxito) |

### Relación con la metodología (TOGAF-ADM adaptado + Design Science Research)

| Fase ADM adaptada | Pasos del notebook |
|---|---|
| Arquitectura de Negocio (requisitos y criterios) | 1 y 2 |
| Arquitectura de Datos (modelo integrado) | 3, 4 y 5 |
| Arquitectura de Aplicaciones (API REST) | 6 y 7 |
| Arquitectura Tecnológica (pruebas y consolidación) | 8, 9 y 10 |

> ⚠️ **Aviso importante sobre los datos.** Los datos reales de SI-01 y SI-02 todavía no están disponibles para este piloto, por lo que aquí se **simulan con las mismas “imperfecciones” típicas** (fechas en formatos distintos, nombres de zona escritos de varias maneras, duplicados, nulos). Los resultados de rendimiento sirven para **validar el diseño y el método**, no como resultado definitivo de la tesis. En el Paso 2 se indica cómo reemplazar los datos simulados por los reales.

### Cómo usarlo
1. Menú **Entorno de ejecución → Ejecutar todo** (tarda aproximadamente 1 minuto).
2. Lee los comentarios de cada paso; las celdas de texto explican *qué* se hace y *por qué*.
3. Al final obtendrás: una base de datos `risktwin_integrado.db`, la especificación `openapi_risktwin.json` y una tabla de veredicto frente a la hipótesis.

## Paso 0 · Preparación del entorno

Instalamos las librerías necesarias. En Colab ya vienen instaladas casi todas; `fastapi` y `uvicorn` suelen faltar.

| Librería | Uso en el piloto |
|---|---|
| `sqlalchemy` | Modelo objeto-relacional y base de datos (SQLite en el piloto) |
| `fastapi` + `uvicorn` | API REST con documentación OpenAPI/Swagger automática |
| `pandas` | Análisis y limpieza de los datos de origen |
| `requests` | Cliente que simula a IND-SIS-02 y SIS-API-01 |

In [ ]:
!pip install -q fastapi "uvicorn[standard]" sqlalchemy requests pandas matplotlib

In [ ]:
import json, re, time, threading, unicodedata, warnings
from datetime import date, datetime
from pathlib import Path

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from matplotlib.patches import FancyBboxPatch
import requests

warnings.filterwarnings("ignore")
pd.set_option("display.max_columns", 30)
pd.set_option("display.width", 150)

SEED = 2026                      # semilla fija → el piloto es reproducible
rng = np.random.default_rng(SEED)
print("Entorno listo ✔")

## Paso 1 · Contexto, criterios de éxito y arquitectura propuesta

**Idea central.** Hoy cada componente guarda la información a su manera. La arquitectura propuesta introduce una **capa de integración** con cuatro piezas: ETL → base de datos centralizada → API REST → consumidores.

**Criterios de éxito** (tomados de la hipótesis del anteproyecto):

- ⏱️ Tiempo de respuesta **inferior a 500 ms** por consulta.
- ✅ Tasa de éxito **superior al 90 %** en las pruebas de integración.

Los dejamos como parámetros para evaluarlos de forma automática en el Paso 8.

In [ ]:
LATENCIA_MAX_MS = 500      # hipótesis: respuesta < 500 ms por consulta
TASA_EXITO_MIN  = 0.90     # hipótesis: tasa de éxito > 90 %

# Catálogo maestro de zonas (clave común que hoy NO comparten SI-01 y SI-02)
CATALOGO_ZONAS = {
    "Z01": "Caldera",
    "Z02": "Almacén de químicos",
    "Z03": "Línea de ensamblaje",
    "Z04": "Soldadura",
    "Z05": "Bodega general",
    "Z06": "Carga y descarga",
}
pd.DataFrame({"codigo": CATALOGO_ZONAS.keys(), "nombre": CATALOGO_ZONAS.values()})

### Diagrama de componentes (documentación OE3)

In [ ]:
def caja(ax, x, y, w, h, texto, color):
    ax.add_patch(FancyBboxPatch((x, y), w, h, boxstyle="round,pad=0.02,rounding_size=0.12",
                                fc=color, ec="#333333", lw=1.3))
    ax.text(x + w/2, y + h/2, texto, ha="center", va="center", fontsize=9.5, linespacing=1.3)

def flecha(ax, x1, y1, x2, y2):
    ax.annotate("", xy=(x2, y2), xytext=(x1, y1),
                arrowprops=dict(arrowstyle="-|>", lw=1.6, color="#333333"))

fig, ax = plt.subplots(figsize=(14, 4.8))
ax.set_xlim(0, 16.2); ax.set_ylim(0, 6); ax.axis("off")

caja(ax, 0.2, 3.9, 2.8, 1.3, "SI-01\nIndicadores de riesgo\n(JSON anidado)", "#cfe8ff")
caja(ax, 0.2, 0.8, 2.8, 1.3, "SI-02\nDataset de incidentes\n(CSV / Excel)",  "#cfe8ff")
caja(ax, 4.0, 2.35, 2.4, 1.3, "Capa ETL\nnormaliza · valida\ncarga", "#ffe9b3")
caja(ax, 7.2, 2.35, 2.4, 1.3, "BD centralizada\nmodelo integrado\n(SQLite → PostgreSQL)", "#d5f5d5")
caja(ax, 10.4, 2.35, 2.4, 1.3, "API REST\nFastAPI + OpenAPI\n(Swagger)", "#f6d5f5")
caja(ax, 13.6, 3.9, 2.4, 1.3, "IND-SIS-02\nDashboard", "#e8e8e8")
caja(ax, 13.6, 0.8, 2.4, 1.3, "SIS-API-01\nGemelo digital\nmultimodal", "#e8e8e8")

flecha(ax, 3.0, 4.5, 4.0, 3.3); flecha(ax, 3.0, 1.5, 4.0, 2.7)
flecha(ax, 6.4, 3.0, 7.2, 3.0); flecha(ax, 9.6, 3.0, 10.4, 3.0)
flecha(ax, 12.8, 3.3, 13.6, 4.4); flecha(ax, 12.8, 2.7, 13.6, 1.6)
ax.set_title("RISKTWIN · Arquitectura de integración propuesta (piloto)", fontsize=12, weight="bold")
plt.tight_layout(); plt.show()

## Paso 2 · Fuentes de datos (simuladas) de SI-01 y SI-02

Para que el piloto se parezca a la realidad, las fuentes simuladas tienen **formatos distintos entre sí**:

| Fuente | Formato simulado | Problemas intencionales |
|---|---|---|
| **SI-02** (incidentes) | CSV tipo “Excel” | Fechas `2026-03-14` y `14/03/2026` mezcladas; zonas escritas como `Caldera`, `CALDERA `, `almacen de quimicos`…; severidad en texto con variantes; duplicados; nulos; y unos pocos registros con errores reales (fecha imposible, zona inexistente) que deben rechazarse |
| **SI-01** (indicadores) | JSON anidado | Código de zona (`Z01`) en lugar del nombre; métricas dentro de un objeto; niveles con mayúsculas distintas |

> 🔁 **Para usar datos reales:** reemplaza las dos celdas de generación por `df_si02_raw = pd.read_csv("tu_archivo_si02.csv", dtype=str)` y `si01_raw = json.load(open("tu_archivo_si01.json"))`. El resto del notebook no cambia.

In [ ]:
# ---------- SI-02 · incidentes (CSV “sucio”) ----------
N_INCIDENTES = 300
tipos_evento = ["Caída", "Golpe", "Quemadura", "Exposición química", "Atrapamiento", "Corte"]
sev_texto    = {1: "Leve", 2: "Moderada", 3: "Grave", 4: "Crítica"}
p_zona       = [0.25, 0.20, 0.10, 0.20, 0.10, 0.15]          # algunas zonas son más riesgosas
codigos      = list(CATALOGO_ZONAS)

def variante_nombre(nombre):
    """Simula cómo distintas personas escriben el mismo nombre de zona."""
    sin_tilde = unicodedata.normalize("NFKD", nombre).encode("ascii", "ignore").decode()
    return rng.choice([nombre, nombre.upper(), nombre.lower(), sin_tilde.lower(), nombre + " "])

def variante_sev(nivel):
    t = sev_texto[nivel]
    return rng.choice([t, t.upper(), t.lower(), unicodedata.normalize("NFKD", t).encode("ascii","ignore").decode()])

inicio, fin = pd.Timestamp("2026-01-01"), pd.Timestamp("2026-08-31")
dias_rango = (fin - inicio).days

filas = []
for i in range(1, N_INCIDENTES + 1):
    zona = rng.choice(codigos, p=p_zona)
    nivel = int(rng.choice([1, 2, 3, 4], p=[0.45, 0.30, 0.18, 0.07]))
    f = inicio + pd.Timedelta(days=int(rng.integers(0, dias_rango)))
    fecha_txt = f.strftime("%Y-%m-%d") if rng.random() < 0.6 else f.strftime("%d/%m/%Y")
    dias_p = {1: 0, 2: int(rng.integers(1, 4)), 3: int(rng.integers(4, 16)), 4: int(rng.integers(16, 61))}[nivel]
    filas.append({
        "id_evento": f"INC-{i:04d}",
        "fecha": fecha_txt,
        "zona": variante_nombre(CATALOGO_ZONAS[zona]),
        "tipo_evento": rng.choice(tipos_evento),
        "severidad": variante_sev(nivel),
        "dias_perdidos": np.nan if rng.random() < 0.05 else dias_p,
    })

df_si02_raw = pd.DataFrame(filas)
# Errores “reales” que el ETL deberá rechazar y registrar (no se corrigen a ciegas)
malos = df_si02_raw.sample(5, random_state=7).index
df_si02_raw.loc[malos[:2], "fecha"] = "31/02/2026"            # fecha imposible
df_si02_raw.loc[malos[2:4], "zona"] = "Patio exterior"        # zona que no existe en el catálogo
df_si02_raw.loc[malos[4], "severidad"] = "Sin clasificar"     # severidad fuera de la escala
# Duplicados “de copia y pega” (muy típicos en hojas de cálculo)
df_si02_raw = pd.concat([df_si02_raw, df_si02_raw.sample(6, random_state=SEED)], ignore_index=True)
df_si02_raw.to_csv("si02_incidentes_simulado.csv", index=False)
df_si02_raw = pd.read_csv("si02_incidentes_simulado.csv", dtype=str)     # se lee como texto, como en la realidad
df_si02_raw.sample(8, random_state=1)

In [ ]:
# ---------- SI-01 · indicadores (JSON anidado) ----------
base_freq = {"Z01": 12, "Z02": 9, "Z03": 5, "Z04": 11, "Z05": 4, "Z06": 7}
base_grav = {"Z01": 30, "Z02": 42, "Z03": 10, "Z04": 28, "Z05": 8, "Z06": 15}
periodos  = [f"2026-{m:02d}" for m in range(1, 9)]

def nivel_desde_puntaje(p):
    return "bajo" if p < 8 else "medio" if p < 18 else "alto" if p < 30 else "critico"

si01_raw = []
for z in codigos:
    for per in periodos:
        frec = max(0.5, rng.normal(base_freq[z], 1.5))
        grav = max(1.0, rng.normal(base_grav[z], 4))
        expo = float(rng.integers(1800, 5200))
        nivel = nivel_desde_puntaje(frec * grav / 20)
        si01_raw.append({
            "zona": z if rng.random() > 0.05 else z.lower() + " ",       # algunos códigos mal escritos
            "periodo": per,
            "metricas": {
                "indice_frecuencia": round(float(frec), 2),
                "indice_gravedad":   round(float(grav), 2),
                "exposicion_horas":  expo,
                "nivel_riesgo": rng.choice([nivel.upper(), nivel.capitalize(), nivel]),
            },
        })

Path("si01_indicadores_simulado.json").write_text(json.dumps(si01_raw, ensure_ascii=False, indent=2))
si01_raw = json.load(open("si01_indicadores_simulado.json"))
print(json.dumps(si01_raw[:2], ensure_ascii=False, indent=2))

## Paso 3 · Análisis de las fuentes (perfilado de datos)

Antes de diseñar el modelo hay que **medir** qué tan fragmentados están los datos. Esta es la parte *cualitativa* del enfoque mixto: producir una **matriz de hallazgos** que justifique cada decisión del modelo integrado (OE1).

In [ ]:
def parsear_fecha(valor):
    for fmt in ("%Y-%m-%d", "%d/%m/%Y"):
        try:
            return datetime.strptime(str(valor).strip(), fmt).date()
        except ValueError:
            continue
    return None

def normalizar_texto(s):
    """minúsculas, sin tildes, sin espacios sobrantes."""
    s = unicodedata.normalize("NFKD", str(s))
    s = "".join(c for c in s if not unicodedata.combining(c))
    return re.sub(r"\s+", " ", s).strip().lower()

fmt_iso   = df_si02_raw["fecha"].str.match(r"^\d{4}-\d{2}-\d{2}$").sum()
fmt_latam = df_si02_raw["fecha"].str.match(r"^\d{2}/\d{2}/\d{4}$").sum()

hallazgos = pd.DataFrame([
    ["SI-02", "Formatos de fecha distintos",           f"{fmt_iso} en AAAA-MM-DD y {fmt_latam} en DD/MM/AAAA",                           "Parser multiformato → tipo DATE"],
    ["SI-02", "Variantes de escritura de zona",        f"{df_si02_raw['zona'].nunique()} textos distintos para {len(CATALOGO_ZONAS)} zonas reales", "Normalizar texto + catálogo maestro de zonas (FK)"],
    ["SI-02", "Severidad como texto con variantes",    f"{df_si02_raw['severidad'].nunique()} variantes para 4 niveles",                 "Escala ordinal 1-4 + etiqueta canónica"],
    ["SI-02", "Registros duplicados",                  f"{df_si02_raw.duplicated('id_evento').sum()} id_evento repetidos",               "Restricción UNIQUE + deduplicación en ETL"],
    ["SI-02", "Valores nulos en días perdidos",        f"{df_si02_raw['dias_perdidos'].isna().sum()} nulos",                             "Columna anulable (no se inventan datos)"],
    ["SI-01", "Zona identificada por código",          "Z01…Z06 (SI-02 usa nombres)",                                                    "Clave común: codigo de zona"],
    ["SI-01", "Estructura anidada (metricas{...})",    "Objeto JSON dentro de cada registro",                                            "Aplanar con json_normalize"],
    ["SI-01", "Nivel de riesgo con mayúsculas mixtas", "ALTO / Alto / alto",                                                             "Valores canónicos en minúscula"],
], columns=["Fuente", "Hallazgo", "Evidencia medida", "Decisión de diseño"])
hallazgos

## Paso 4 · Modelo de datos integrado (objeto-relacional)

El modelo gira alrededor de la **zona**, que es la clave que une ambos mundos:

- `zona` (catálogo maestro) **1 — N** `incidente` (viene de SI-02)
- `zona` (catálogo maestro) **1 — N** `indicador_riesgo` (viene de SI-01, uno por zona y periodo)

Decisiones de diseño: claves foráneas, restricciones `UNIQUE` para evitar duplicados, un campo `fuente` para trazabilidad y un índice por `(zona, fecha)` para acelerar las consultas del dashboard.

In [ ]:
from sqlalchemy import (create_engine, String, Integer, Float, Date, ForeignKey,
                        UniqueConstraint, Index, select, func, text)
from sqlalchemy.orm import DeclarativeBase, Mapped, mapped_column, sessionmaker
from typing import Optional

class Base(DeclarativeBase):
    pass

class Zona(Base):
    __tablename__ = "zona"
    codigo: Mapped[str] = mapped_column(String(8), primary_key=True)
    nombre: Mapped[str] = mapped_column(String(80), unique=True)

class Incidente(Base):
    __tablename__ = "incidente"
    id: Mapped[int] = mapped_column(Integer, primary_key=True, autoincrement=True)
    id_evento: Mapped[str] = mapped_column(String(20), unique=True)
    fecha: Mapped[date] = mapped_column(Date)
    zona_codigo: Mapped[str] = mapped_column(ForeignKey("zona.codigo"))
    tipo_evento: Mapped[str] = mapped_column(String(60))
    severidad_nivel: Mapped[int] = mapped_column(Integer)
    severidad_etiqueta: Mapped[str] = mapped_column(String(12))
    dias_perdidos: Mapped[Optional[int]] = mapped_column(Integer, nullable=True)
    fuente: Mapped[str] = mapped_column(String(10), default="SI-02")
    __table_args__ = (Index("ix_incidente_zona_fecha", "zona_codigo", "fecha"),)

class IndicadorRiesgo(Base):
    __tablename__ = "indicador_riesgo"
    id: Mapped[int] = mapped_column(Integer, primary_key=True, autoincrement=True)
    zona_codigo: Mapped[str] = mapped_column(ForeignKey("zona.codigo"))
    periodo: Mapped[str] = mapped_column(String(7))                 # AAAA-MM
    indice_frecuencia: Mapped[float] = mapped_column(Float)
    indice_gravedad: Mapped[float] = mapped_column(Float)
    exposicion_horas: Mapped[float] = mapped_column(Float)
    nivel_riesgo: Mapped[str] = mapped_column(String(10))
    fuente: Mapped[str] = mapped_column(String(10), default="SI-01")
    __table_args__ = (UniqueConstraint("zona_codigo", "periodo", name="uq_zona_periodo"),)

DB_PATH = Path("risktwin_integrado.db")
if DB_PATH.exists():
    DB_PATH.unlink()                                        # empezamos limpio en cada ejecución
engine = create_engine(f"sqlite:///{DB_PATH}", connect_args={"check_same_thread": False})
Base.metadata.create_all(engine)
SessionLocal = sessionmaker(bind=engine, expire_on_commit=False)

with engine.connect() as con:
    tablas = con.execute(text("SELECT name FROM sqlite_master WHERE type='table' ORDER BY name")).scalars().all()
print("Tablas creadas:", tablas)

### Diagrama entidad-relación (documentación OE3)

In [ ]:
def entidad(ax, x, y, titulo, columnas, color, ancho=3.6):
    alto = 0.42 * (len(columnas) + 1.4)
    ax.add_patch(FancyBboxPatch((x, y - alto), ancho, alto, boxstyle="round,pad=0.02,rounding_size=0.08",
                                fc="white", ec="#333", lw=1.3))
    ax.add_patch(FancyBboxPatch((x, y - 0.5), ancho, 0.5, boxstyle="round,pad=0.02,rounding_size=0.08",
                                fc=color, ec="#333", lw=1.3))
    ax.text(x + ancho/2, y - 0.25, titulo, ha="center", va="center", weight="bold", fontsize=10.5)
    for i, c in enumerate(columnas):
        ax.text(x + 0.15, y - 0.85 - i * 0.42, c, ha="left", va="center", fontsize=8.6, family="monospace")
    return (x, y - alto, ancho, alto)

fig, ax = plt.subplots(figsize=(14, 4.8))
ax.set_xlim(0, 15); ax.set_ylim(2.5, 6.8); ax.axis("off")

entidad(ax, 0.3, 6.5, "incidente  (SI-02)", [
    "PK id", "UQ id_evento", "   fecha", "FK zona_codigo", "   tipo_evento", "   severidad_nivel (1-4)",
    "   severidad_etiqueta", "   dias_perdidos (null)", "   fuente"], "#cfe8ff")
entidad(ax, 5.7, 4.3, "zona  (catálogo)", ["PK codigo", "UQ nombre"], "#d5f5d5")
entidad(ax, 10.9, 6.5, "indicador_riesgo  (SI-01)", [
    "PK id", "FK zona_codigo", "   periodo (AAAA-MM)", "   indice_frecuencia", "   indice_gravedad",
    "   exposicion_horas", "   nivel_riesgo", "   fuente", "UQ (zona_codigo, periodo)"], "#ffe9b3", ancho=3.8)

ax.annotate("", xy=(5.7, 3.45), xytext=(3.9, 3.45), arrowprops=dict(arrowstyle="-", lw=1.6))
ax.text(4.8, 3.6, "N ── 1", ha="center", fontsize=9)
ax.annotate("", xy=(9.3, 3.45), xytext=(10.9, 3.45), arrowprops=dict(arrowstyle="-", lw=1.6))
ax.text(10.1, 3.6, "1 ── N", ha="center", fontsize=9)
ax.set_title("Modelo entidad-relación integrado · RISKTWIN (piloto)", fontsize=12, weight="bold")
plt.tight_layout(); plt.show()

## Paso 5 · ETL: normalizar, validar y cargar en la base centralizada

Cada fuente pasa por una función de normalización que devuelve **(datos válidos, registro de calidad)**. Nada se descarta en silencio: todo lo rechazado queda anotado con su motivo. Esto es lo que permite demostrar que la integración es confiable.

In [ ]:
MAPA_ZONAS = {normalizar_texto(nombre): cod for cod, nombre in CATALOGO_ZONAS.items()}
MAPA_SEV   = {"leve": 1, "moderada": 2, "grave": 3, "critica": 4}
ETIQUETA   = {1: "Leve", 2: "Moderada", 3: "Grave", 4: "Crítica"}
MAPA_TIPOS = {normalizar_texto(t): t for t in tipos_evento}
NIVELES_OK = {"bajo", "medio", "alto", "critico"}

def etl_si02(df_raw):
    df, log = df_raw.copy(), []

    n0 = len(df)
    df = df.drop_duplicates("id_evento", keep="first")
    log.append(("SI-02", "duplicado (id_evento repetido)", n0 - len(df)))

    df["fecha_n"] = df["fecha"].map(parsear_fecha)
    log.append(("SI-02", "fecha inválida", int(df["fecha_n"].isna().sum())))

    df["zona_codigo"] = df["zona"].map(lambda s: MAPA_ZONAS.get(normalizar_texto(s)))
    log.append(("SI-02", "zona no reconocida en el catálogo", int(df["zona_codigo"].isna().sum())))

    df["sev_nivel"] = df["severidad"].map(lambda s: MAPA_SEV.get(normalizar_texto(s)))
    log.append(("SI-02", "severidad no reconocida", int(df["sev_nivel"].isna().sum())))

    df["tipo_n"] = df["tipo_evento"].map(lambda s: MAPA_TIPOS.get(normalizar_texto(s)))
    log.append(("SI-02", "tipo de evento no reconocido", int(df["tipo_n"].isna().sum())))

    ok = df.dropna(subset=["fecha_n", "zona_codigo", "sev_nivel", "tipo_n"]).copy()
    ok["sev_nivel"] = ok["sev_nivel"].astype(int)
    ok["dias_perdidos_n"] = pd.to_numeric(ok["dias_perdidos"], errors="coerce")
    return ok, pd.DataFrame(log, columns=["fuente", "motivo", "registros"])

def etl_si01(raw_json):
    df = pd.json_normalize(raw_json)                                    # aplana metricas.*
    df.columns = [c.replace("metricas.", "") for c in df.columns]
    log = []

    df["zona_codigo"] = df["zona"].astype(str).str.strip().str.upper()
    log.append(("SI-01", "código de zona corregido (espacios/minúsculas)", int((df["zona_codigo"] != df["zona"]).sum())))
    invalid_zone = ~df["zona_codigo"].isin(CATALOGO_ZONAS)
    log.append(("SI-01", "zona no reconocida en el catálogo", int(invalid_zone.sum())))

    df["nivel_n"] = df["nivel_riesgo"].map(normalizar_texto)
    bad_level = ~df["nivel_n"].isin(NIVELES_OK)
    log.append(("SI-01", "nivel de riesgo no reconocido", int(bad_level.sum())))

    neg = (df[["indice_frecuencia", "indice_gravedad", "exposicion_horas"]] < 0).any(axis=1)
    log.append(("SI-01", "valores negativos", int(neg.sum())))

    ok = df[~invalid_zone & ~bad_level & ~neg].drop_duplicates(["zona_codigo", "periodo"]).copy()
    return ok, pd.DataFrame(log, columns=["fuente", "motivo", "registros"])

si02_ok, log02 = etl_si02(df_si02_raw)
si01_ok, log01 = etl_si01(si01_raw)
registro_calidad = pd.concat([log02, log01], ignore_index=True)
registro_calidad

In [ ]:
# ---------- Carga en la base de datos ----------
with SessionLocal() as s:
    s.add_all([Zona(codigo=c, nombre=n) for c, n in CATALOGO_ZONAS.items()])
    s.flush()
    s.add_all([
        Incidente(id_evento=r.id_evento, fecha=r.fecha_n, zona_codigo=r.zona_codigo, tipo_evento=r.tipo_n,
                  severidad_nivel=int(r.sev_nivel), severidad_etiqueta=ETIQUETA[int(r.sev_nivel)],
                  dias_perdidos=None if pd.isna(r.dias_perdidos_n) else int(r.dias_perdidos_n))
        for r in si02_ok.itertuples()
    ])
    s.add_all([
        IndicadorRiesgo(zona_codigo=r.zona_codigo, periodo=r.periodo, indice_frecuencia=r.indice_frecuencia,
                        indice_gravedad=r.indice_gravedad, exposicion_horas=r.exposicion_horas, nivel_riesgo=r.nivel_n)
        for r in si01_ok.itertuples()
    ])
    s.commit()

# ---------- Verificaciones de integridad ----------
with engine.connect() as con:
    n_inc = con.execute(text("SELECT COUNT(*) FROM incidente")).scalar()
    n_ind = con.execute(text("SELECT COUNT(*) FROM indicador_riesgo")).scalar()
    huerfanos = con.execute(text("PRAGMA foreign_key_check")).fetchall()

print(f"Incidentes cargados      : {n_inc}  (de {len(df_si02_raw)} filas de origen)")
print(f"Indicadores cargados     : {n_ind}  (de {len(si01_raw)} registros de origen)")
print(f"Violaciones de FK        : {len(huerfanos)}")
assert n_inc == len(si02_ok) and n_ind == len(si01_ok) and len(huerfanos) == 0
print("Integridad referencial ✔")

## Paso 6 · API REST con FastAPI

La API es la **única puerta de entrada** a los datos integrados. Los consumidores (dashboard y gemelo digital) no tocan la base de datos directamente.

| Endpoint | Consumidor principal | Qué entrega |
|---|---|---|
| `GET /health` | Monitoreo | Estado del servicio |
| `GET /zonas` | Ambos | Catálogo de zonas |
| `GET /incidentes` | IND-SIS-02 | Incidentes con filtros (zona, fechas, severidad mínima, paginación) |
| `GET /indicadores` | IND-SIS-02 | Indicadores de riesgo por zona y periodo |
| `GET /riesgo/zonas` | IND-SIS-02 | **Vista integrada** SI-01 + SI-02 por zona |
| `GET /gemelo/estado` | SIS-API-01 | Estado compacto por zona para alimentar el gemelo digital |

Los modelos de respuesta (Pydantic) hacen que FastAPI genere **automáticamente** la especificación OpenAPI y la interfaz Swagger.

In [ ]:
from fastapi import FastAPI, Depends, HTTPException, Query
from pydantic import BaseModel, ConfigDict, Field

# ---------- Esquemas de respuesta (contrato de datos) ----------
class ZonaOut(BaseModel):
    model_config = ConfigDict(from_attributes=True)
    codigo: str = Field(examples=["Z01"])
    nombre: str = Field(examples=["Caldera"])

class IncidenteOut(BaseModel):
    id_evento: str
    fecha: date
    zona_codigo: str
    zona_nombre: str
    tipo_evento: str
    severidad_nivel: int = Field(ge=1, le=4)
    severidad_etiqueta: str
    dias_perdidos: Optional[int] = None

class IndicadorOut(BaseModel):
    model_config = ConfigDict(from_attributes=True)
    zona_codigo: str
    periodo: str
    indice_frecuencia: float
    indice_gravedad: float
    exposicion_horas: float
    nivel_riesgo: str

class RiesgoZonaOut(BaseModel):
    zona_codigo: str
    zona_nombre: str
    total_incidentes: int
    severidad_promedio: Optional[float]
    dias_perdidos_total: int
    periodo_indicador: Optional[str]
    indice_frecuencia: Optional[float]
    indice_gravedad: Optional[float]
    exposicion_horas: Optional[float]
    nivel_riesgo: Optional[str]

class ZonaEstadoGemelo(BaseModel):
    zona: str
    nivel_riesgo: Optional[str]
    incidentes_acumulados: int
    severidad_promedio: Optional[float]

class GemeloEstadoOut(BaseModel):
    generado_en: datetime
    version_modelo: str
    zonas: list[ZonaEstadoGemelo]

# ---------- Aplicación ----------
app = FastAPI(
    title="RISKTWIN · API de integración de datos de riesgo industrial",
    version="0.1.0-piloto",
    description="Capa de integración que unifica los datos de SI-01 (indicadores) y SI-02 (incidentes) "
                "para su consumo por IND-SIS-02 (dashboard) y SIS-API-01 (gemelo digital multimodal).",
)

def get_db():
    db = SessionLocal()
    try:
        yield db
    finally:
        db.close()

@app.get("/health", tags=["Sistema"], summary="Estado del servicio")
def health():
    return {"status": "ok", "hora_servidor": datetime.now().isoformat(timespec="seconds")}

@app.get("/zonas", response_model=list[ZonaOut], tags=["Catálogo"], summary="Listar zonas")
def listar_zonas(db=Depends(get_db)):
    return db.scalars(select(Zona).order_by(Zona.codigo)).all()

@app.get("/incidentes", response_model=list[IncidenteOut], tags=["SI-02 · Incidentes"],
         summary="Consultar incidentes con filtros")
def listar_incidentes(
    zona: Optional[str] = Query(None, description="Código de zona, p. ej. Z01"),
    desde: Optional[date] = Query(None, description="Fecha inicial (AAAA-MM-DD)"),
    hasta: Optional[date] = Query(None, description="Fecha final (AAAA-MM-DD)"),
    severidad_min: int = Query(1, ge=1, le=4, description="Severidad mínima (1 leve … 4 crítica)"),
    limit: int = Query(50, ge=1, le=500),
    offset: int = Query(0, ge=0),
    db=Depends(get_db),
):
    if zona and not db.get(Zona, zona):
        raise HTTPException(status_code=404, detail=f"Zona '{zona}' no existe en el catálogo")
    q = (select(Incidente, Zona.nombre).join(Zona, Zona.codigo == Incidente.zona_codigo)
         .where(Incidente.severidad_nivel >= severidad_min))
    if zona:   q = q.where(Incidente.zona_codigo == zona)
    if desde:  q = q.where(Incidente.fecha >= desde)
    if hasta:  q = q.where(Incidente.fecha <= hasta)
    q = q.order_by(Incidente.fecha.desc(), Incidente.id_evento).limit(limit).offset(offset)
    return [IncidenteOut(id_evento=i.id_evento, fecha=i.fecha, zona_codigo=i.zona_codigo, zona_nombre=n,
                         tipo_evento=i.tipo_evento, severidad_nivel=i.severidad_nivel,
                         severidad_etiqueta=i.severidad_etiqueta, dias_perdidos=i.dias_perdidos)
            for i, n in db.execute(q).all()]

@app.get("/indicadores", response_model=list[IndicadorOut], tags=["SI-01 · Indicadores"],
         summary="Consultar indicadores de riesgo")
def listar_indicadores(zona: Optional[str] = None, periodo: Optional[str] = Query(None, pattern=r"^\d{4}-\d{2}$"),
                       db=Depends(get_db)):
    if zona and not db.get(Zona, zona):
        raise HTTPException(status_code=404, detail=f"Zona '{zona}' no existe en el catálogo")
    q = select(IndicadorRiesgo).order_by(IndicadorRiesgo.zona_codigo, IndicadorRiesgo.periodo)
    if zona:    q = q.where(IndicadorRiesgo.zona_codigo == zona)
    if periodo: q = q.where(IndicadorRiesgo.periodo == periodo)
    return db.scalars(q).all()

def _vista_integrada(db) -> list[RiesgoZonaOut]:
    agg = {r.zona_codigo: r for r in db.execute(
        select(Incidente.zona_codigo,
               func.count(Incidente.id).label("n"),
               func.avg(Incidente.severidad_nivel).label("sev"),
               func.coalesce(func.sum(Incidente.dias_perdidos), 0).label("dias"))
        .group_by(Incidente.zona_codigo)).all()}
    ultimo = (select(IndicadorRiesgo.zona_codigo, func.max(IndicadorRiesgo.periodo).label("p"))
              .group_by(IndicadorRiesgo.zona_codigo).subquery())
    ind = {i.zona_codigo: i for i in db.scalars(
        select(IndicadorRiesgo).join(ultimo, (IndicadorRiesgo.zona_codigo == ultimo.c.zona_codigo)
                                     & (IndicadorRiesgo.periodo == ultimo.c.p))).all()}
    salida = []
    for z in db.scalars(select(Zona).order_by(Zona.codigo)).all():
        a, i = agg.get(z.codigo), ind.get(z.codigo)
        salida.append(RiesgoZonaOut(
            zona_codigo=z.codigo, zona_nombre=z.nombre,
            total_incidentes=int(a.n) if a else 0,
            severidad_promedio=round(float(a.sev), 2) if a else None,
            dias_perdidos_total=int(a.dias) if a else 0,
            periodo_indicador=i.periodo if i else None,
            indice_frecuencia=i.indice_frecuencia if i else None,
            indice_gravedad=i.indice_gravedad if i else None,
            exposicion_horas=i.exposicion_horas if i else None,
            nivel_riesgo=i.nivel_riesgo if i else None))
    return salida

@app.get("/riesgo/zonas", response_model=list[RiesgoZonaOut], tags=["Vista integrada"],
         summary="Riesgo por zona (SI-01 + SI-02 integrados)")
def riesgo_por_zona(db=Depends(get_db)):
    return _vista_integrada(db)

@app.get("/gemelo/estado", response_model=GemeloEstadoOut, tags=["Gemelo digital"],
         summary="Estado compacto por zona para SIS-API-01")
def estado_gemelo(db=Depends(get_db)):
    return GemeloEstadoOut(
        generado_en=datetime.now(), version_modelo="0.1.0-piloto",
        zonas=[ZonaEstadoGemelo(zona=r.zona_codigo, nivel_riesgo=r.nivel_riesgo,
                                incidentes_acumulados=r.total_incidentes,
                                severidad_promedio=r.severidad_promedio) for r in _vista_integrada(db)])

print("Rutas registradas:", [r.path for r in app.routes if r.path.startswith("/") and r.path not in ("/openapi.json", "/docs", "/docs/oauth2-redirect", "/redoc")])

## Paso 7 · Levantar el servicio y documentación OpenAPI/Swagger

Arrancamos la API en un **hilo en segundo plano** (así el notebook sigue disponible) y descargamos su especificación OpenAPI. Esa especificación es el *contrato* que los equipos de IND-SIS-02 y SIS-API-01 pueden consumir (OE3).

In [ ]:
import uvicorn

PUERTO = 8000
URL = f"http://127.0.0.1:{PUERTO}"

server = uvicorn.Server(uvicorn.Config(app, host="127.0.0.1", port=PUERTO, log_level="warning"))
hilo = threading.Thread(target=server.run, daemon=True)
hilo.start()

for _ in range(50):                                   # esperamos hasta 10 s a que arranque
    try:
        if requests.get(f"{URL}/health", timeout=1).status_code == 200:
            break
    except requests.ConnectionError:
        time.sleep(0.2)
print("Servicio activo en", URL, "→", requests.get(f"{URL}/health").json())

In [ ]:
openapi = requests.get(f"{URL}/openapi.json").json()
Path("openapi_risktwin.json").write_text(json.dumps(openapi, ensure_ascii=False, indent=2))

tabla_api = pd.DataFrame([
    {"método": m.upper(), "ruta": ruta, "resumen": d.get("summary"), "etiqueta": d["tags"][0]}
    for ruta, ops in openapi["paths"].items() for m, d in ops.items()
])
print(f"Especificación {openapi['openapi']} · {openapi['info']['title']} · v{openapi['info']['version']}")
tabla_api

In [ ]:
# Interfaz Swagger embebida (solo funciona dentro de Colab; fuera de Colab, abre http://127.0.0.1:8000/docs)
try:
    from google.colab import output
    output.serve_kernel_port_as_iframe(PUERTO, path="/docs", height=620)
except ImportError:
    print("No estás en Colab: abre", f"{URL}/docs", "en tu navegador.")

## Paso 8 · Pruebas de integración y rendimiento (OE4)

Este paso es el equivalente en Python de lo que el anteproyecto plantea hacer con **Postman y JMeter**:

1. Definimos un **plan de pruebas**: 10 casos (8 consultas válidas y 2 casos de error que *deben* fallar con el código correcto).
2. Repetimos cada caso 30 veces → **300 solicitudes**.
3. Medimos el tiempo de cada solicitud y decidimos si fue **exitosa**: código HTTP esperado **y** cuerpo con la estructura correcta.
4. Comparamos con la hipótesis: **latencia < 500 ms** y **tasa de éxito > 90 %**.

> Se reporta la **media, la mediana y el percentil 95 (p95)**. El p95 es más exigente que la media: indica que el 95 % de las consultas respondió por debajo de ese valor.

In [ ]:
REPETICIONES = 30

casos = [
    # (nombre,                       ruta,            parámetros,                                         status esperado, claves obligatorias)
    ("health",                       "/health",        None,                                               200, {"status"}),
    ("zonas",                        "/zonas",         None,                                               200, {"codigo", "nombre"}),
    ("incidentes (paginados)",       "/incidentes",    {"limit": 50},                                      200, {"id_evento", "zona_nombre"}),
    ("incidentes (zona+severidad)",  "/incidentes",    {"zona": "Z01", "severidad_min": 3},                200, {"id_evento"}),
    ("incidentes (rango fechas)",    "/incidentes",    {"desde": "2026-03-01", "hasta": "2026-05-31"},     200, {"id_evento"}),
    ("indicadores (zona)",           "/indicadores",   {"zona": "Z02"},                                    200, {"periodo", "nivel_riesgo"}),
    ("riesgo integrado",             "/riesgo/zonas",  None,                                               200, {"zona_codigo", "total_incidentes", "nivel_riesgo"}),
    ("estado gemelo",                "/gemelo/estado", None,                                               200, {"zonas", "generado_en"}),
    ("ERROR esperado: zona inexistente",    "/incidentes", {"zona": "ZXX"},                                404, {"detail"}),
    ("ERROR esperado: parámetro inválido",  "/incidentes", {"severidad_min": 9},                           422, {"detail"}),
]

def cuerpo_valido(payload, claves):
    filas = payload if isinstance(payload, list) else [payload]
    if not filas:
        return False
    return all(claves.issubset(f.keys()) for f in filas)

sesion = requests.Session()
for _ in range(5):                                      # calentamiento (no se mide)
    sesion.get(f"{URL}/riesgo/zonas")

resultados = []
for nombre, ruta, params, esperado, claves in casos:
    for rep in range(REPETICIONES):
        t0 = time.perf_counter()
        r = sesion.get(f"{URL}{ruta}", params=params, timeout=10)
        ms = (time.perf_counter() - t0) * 1000
        try:
            ok = (r.status_code == esperado) and cuerpo_valido(r.json(), claves)
        except ValueError:
            ok = False
        resultados.append({"caso": nombre, "ruta": ruta, "status": r.status_code, "exito": ok, "ms": ms})

df_res = pd.DataFrame(resultados)
print(f"Solicitudes ejecutadas: {len(df_res)}")

In [ ]:
resumen = (df_res.groupby("caso", sort=False)
           .agg(solicitudes=("ms", "size"),
                exito_pct=("exito", lambda s: 100 * s.mean()),
                media_ms=("ms", "mean"),
                mediana_ms=("ms", "median"),
                p95_ms=("ms", lambda s: s.quantile(0.95)),
                max_ms=("ms", "max"))
           .round(2))
resumen

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(14, 4.6))

orden = list(resumen.index)
axes[0].boxplot([df_res[df_res.caso == c].ms for c in orden], vert=False, showfliers=True)
axes[0].set_yticks(range(1, len(orden) + 1)); axes[0].set_yticklabels(orden, fontsize=8.5)
axes[0].axvline(LATENCIA_MAX_MS, color="crimson", ls="--", label=f"umbral {LATENCIA_MAX_MS} ms")
axes[0].set_xscale("log"); axes[0].set_xlabel("tiempo de respuesta (ms, escala log)")
axes[0].set_title("Latencia por caso de prueba"); axes[0].legend(); axes[0].invert_yaxis()

bins = np.logspace(np.log10(df_res.ms.min() * 0.9), np.log10(max(df_res.ms.max(), LATENCIA_MAX_MS) * 1.1), 40)
axes[1].hist(df_res.ms, bins=bins, color="#4c78a8"); axes[1].set_xscale("log")
axes[1].axvline(df_res.ms.quantile(0.95), color="orange", ls="--", label=f"p95 = {df_res.ms.quantile(0.95):.1f} ms")
axes[1].axvline(LATENCIA_MAX_MS, color="crimson", ls="--", label=f"umbral {LATENCIA_MAX_MS} ms")
axes[1].set_xlabel("ms (escala log)"); axes[1].set_ylabel("solicitudes"); axes[1].set_title("Distribución global"); axes[1].legend()
plt.tight_layout(); plt.show()

In [ ]:
# ---------- Prueba de concurrencia: dashboard y gemelo digital consultando a la vez ----------
from concurrent.futures import ThreadPoolExecutor

def llamada(_):
    t0 = time.perf_counter()
    r = requests.get(f"{URL}/riesgo/zonas", timeout=10)
    return r.status_code == 200, (time.perf_counter() - t0) * 1000

with ThreadPoolExecutor(max_workers=8) as pool:
    res_conc = list(pool.map(llamada, range(200)))

conc = pd.DataFrame(res_conc, columns=["exito", "ms"])
print(f"Concurrencia (8 clientes simultáneos, 200 solicitudes): éxito {100*conc.exito.mean():.1f} % · "
      f"media {conc.ms.mean():.1f} ms · p95 {conc.ms.quantile(.95):.1f} ms")

### Veredicto frente a la hipótesis

In [ ]:
tasa_exito = df_res.exito.mean()
validas = df_res[~df_res.caso.str.startswith("ERROR")]
p95_global = df_res.ms.quantile(0.95)
media_global = df_res.ms.mean()

veredicto = pd.DataFrame([
    ["Tiempo de respuesta p95 < 500 ms",      f"{p95_global:.1f} ms",      p95_global < LATENCIA_MAX_MS],
    ["Tiempo de respuesta medio < 500 ms",    f"{media_global:.1f} ms",    media_global < LATENCIA_MAX_MS],
    ["Tasa de éxito > 90 %",                  f"{100*tasa_exito:.1f} %",   tasa_exito > TASA_EXITO_MIN],
    ["Éxito en consultas válidas",            f"{100*validas.exito.mean():.1f} %", validas.exito.mean() > TASA_EXITO_MIN],
    ["Integridad referencial en la BD",       "0 violaciones de FK",       len(huerfanos) == 0],
], columns=["Criterio", "Resultado medido", "¿Cumple?"])
veredicto["¿Cumple?"] = veredicto["¿Cumple?"].map({True: "✅ Sí", False: "❌ No"})
veredicto

## Paso 9 · Simulación de los consumidores: IND-SIS-02 y SIS-API-01

La tesis afirma que la arquitectura permite que **otros componentes consuman** la información. Lo comprobamos con dos clientes mínimos que usan *solo la API* (nunca la base de datos):

- **IND-SIS-02 (dashboard):** pide `/riesgo/zonas` y dibuja un gráfico por zona.
- **SIS-API-01 (gemelo digital):** pide `/gemelo/estado` y recibe un estado compacto listo para alimentar el modelo.

In [ ]:
# ----- Cliente 1: dashboard (IND-SIS-02) -----
riesgo = pd.DataFrame(requests.get(f"{URL}/riesgo/zonas").json())
colores = {"bajo": "#8fce8f", "medio": "#f4d35e", "alto": "#f4a259", "critico": "#d1495b"}

fig, ax = plt.subplots(figsize=(10, 4.3))
riesgo_ord = riesgo.sort_values("total_incidentes", ascending=False)
ax.bar(riesgo_ord.zona_nombre, riesgo_ord.total_incidentes,
       color=[colores.get(n, "#bbbbbb") for n in riesgo_ord.nivel_riesgo], edgecolor="#333")
for x, (n, sev) in enumerate(zip(riesgo_ord.total_incidentes, riesgo_ord.severidad_promedio)):
    ax.text(x, n + 0.8, f"sev. prom. {sev}", ha="center", fontsize=8.5)
ax.set_ylabel("incidentes acumulados (SI-02)")
ax.set_title("IND-SIS-02 (simulado) · Incidentes por zona, coloreados por nivel de riesgo de SI-01")
plt.xticks(rotation=20); plt.tight_layout(); plt.show()
riesgo

In [ ]:
# ----- Cliente 2: gemelo digital (SIS-API-01) -----
estado = requests.get(f"{URL}/gemelo/estado").json()
print(json.dumps(estado, ensure_ascii=False, indent=2)[:900], "\n…")

## Paso 10 · Conclusiones del piloto, limitaciones y próximos pasos

**Qué quedó demostrado**
- Es viable unificar fuentes con formatos distintos en un **modelo de datos común** con integridad referencial.
- Una **API REST documentada con OpenAPI** permite que IND-SIS-02 y SIS-API-01 consuman la información sin conocer la base de datos.
- El método de evaluación (latencia y tasa de éxito con criterios explícitos) es **automático y repetible**.

**Limitaciones (conviene decirlas en la tesis)**
- Los datos son **simulados**; los tiempos con datos reales y mayor volumen pueden ser distintos.
- SQLite es ideal para el piloto, pero la versión final debería usar un motor cliente-servidor (p. ej. **PostgreSQL**) y pruebas con **JMeter** bajo carga sostenida.
- No se implementó autenticación ni control de acceso, ni carga incremental (solo carga completa).

**Próximos pasos sugeridos**
1. Reemplazar las fuentes simuladas por los archivos reales de SI-01 y SI-02 (Paso 2) y revisar la matriz de hallazgos del Paso 3.
2. Acordar con los equipos de SI-01, SI-02, IND-SIS-02 y SIS-API-01 el **contrato OpenAPI** (`openapi_risktwin.json`).
3. Migrar la base a PostgreSQL y repetir el Paso 8 con el volumen real.
4. Exportar los diagramas y la matriz de hallazgos como anexos de la tesis.

In [ ]:
# Cierre ordenado del servicio y listado de entregables generados
server.should_exit = True
hilo.join(timeout=5)

print("Servicio detenido ✔\n")
print("Archivos generados en el piloto:")
for f in ["si02_incidentes_simulado.csv", "si01_indicadores_simulado.json", "risktwin_integrado.db", "openapi_risktwin.json"]:
    print(f"  • {f:35s} {Path(f).stat().st_size/1024:8.1f} KB")

# En Colab puedes descargarlos con:
# from google.colab import files; files.download("openapi_risktwin.json")